# Project Milestone 4

In [148]:
pip install pyspark

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 434.1/434.1 MB 3.0 MB/s eta 0:00:0000:0100:05
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.0/203.0 kB 2.5 MB/s eta 0:00:00a 0:00:01
  Created wheel for pyspark: filename=pyspark-4.0.0-py2.py3-none-any.whl size=434741237 sha256=914e45d49bfa129240bb857adf5a868292ccec075db8c8a0c5830d0be01b4c48
  Stored in directory: /Users/vijsharm/Library/Caches/pip/wheels/2d/77/9b/12660be70f7f447940a0caede37ae208b2e0d1c8487dce52a6
Successfully built pyspark
Note: you may need to restart the kernel to use updated packages.


In [77]:
# Importing Pandas packages required for this excercise.

import pandas as pd
from urllib.request import urlopen
import json

import urllib.request, urllib.parse, urllib.error
import requests
from bs4 import BeautifulSoup
import ssl
import re

import matplotlib.dates as mdates
from matplotlib.dates import DateFormatter
import matplotlib.ticker as tick


In [79]:
def read_json_from_web_api(url):
    """
    Reads a JSON file from a web API.

    Args:
        url (str): The URL of the web API endpoint.

    Returns:
        dict: The JSON data as a Python dictionary, or None if an error occurs.
    """
    try:
        response = requests.get(url)
        response.raise_for_status()  # Raise HTTPError for bad responses (4xx or 5xx)
        return response.json()
    except requests.exceptions.RequestException as e:
        print(f"Request error: {e}")
        return None
    except json.JSONDecodeError as e:
        print(f"JSON decode error: {e}")
        return None

In [107]:
# call Covid 19 API and load it in Json Format
url = "https://api.covidtracking.com/v1/us/daily.json"  # Replace with the actual API endpoint
data = read_json_from_web_api(url)
df = pd.json_normalize(data)
df = df.drop('hash',axis=1)    
df.head(10)

,date,states,positive,negative,pending,hospitalizedCurrently,hospitalizedCumulative,inIcuCurrently,inIcuCumulative,onVentilatorCurrently,...,totalTestResults,lastModified,recovered,total,posNeg,deathIncrease,hospitalizedIncrease,negativeIncrease,positiveIncrease,totalTestResultsIncrease
0,20210307,56,28756489.0,74582825.0,11808.0,40199.0,776361.0,8134.0,45475.0,2802.0,...,363825123,2021-03-07T24:00:00Z,None,0,0,842,726,131835,41835,1170059
1,20210306,56,28714654.0,74450990.0,11783.0,41401.0,775635.0,8409.0,45453.0,2811.0,...,362655064,2021-03-06T24:00:00Z,None,0,0,1680,503,143835,60015,1430992
2,20210305,56,28654639.0,74307155.0,12213.0,42541.0,775132.0,8634.0,45373.0,2889.0,...,361224072,2021-03-05T24:00:00Z,None,0,0,2221,2781,271917,68787,1744417
3,20210304,56,28585852.0,74035238.0,12405.0,44172.0,772351.0,8970.0,45293.0,2973.0,...,359479655,2021-03-04T24:00:00Z,None,0,0,1743,1530,177957,65487,1590984
4,20210303,56,28520365.0,73857281.0,11778.0,45462.0,770821.0,9359.0,45214.0,3094.0,...,357888671,2021-03-03T24:00:00Z,None,0,0,2449,2172,267001,66836,1406795
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,20201202,56,13925720.0,50105551.0,14368.0,100327.0,467773.0,19687.0,31038.0,6855.0,...,201554613,2020-12-02T24:00:00Z,None,0,0,2811,5238,-658774,203429,1587969
96,20201201,56,13722291.0,50764325.0,8764.0,98814.0,462535.0,19302.0,30749.0,6651.0,...,199966644,2020-12-01T24:00:00Z,None,0,0,2489,4916,263699,181183,1494046
97,20201130,56,13541108.0,50500626.0,14883.0,96134.0,457619.0,18807.0,30469.0,6520.0,...,198472598,2020-11-30T24:00:00Z,None,0,0,1037,3394,344231,150031,1520240
98,20201129,56,13391077.0,50156395.0,14860.0,93357.0,454225.0,18437.0,30274.0,6245.0,...,196952358,2020-11-29T24:00:00Z,None,0,0,825,2429,226788,137254,1337135


### Step 1. Dropping Duplicates from dataframe

In [119]:
pd.set_option('display.max_rows', 180)
(df
    .filter(['date'])
    .drop_duplicates()
)

,date
0,20210307
1,20210306
2,20210305
3,20210304
4,20210303
...,...
415,20200117
416,20200116
417,20200115
418,20200114


### Step 2. Format Date and find Total number of negativeIncrease by month

In [179]:
df['date_new'] = pd.to_datetime(df['date'], format='%Y%m%d')

monthly_sum = df.groupby(pd.Grouper(key='date_new', freq='ME'))['negativeIncrease'].sum()

print(monthly_sum)

date_new
2020-01-31          0
2020-02-29          0
2020-03-31     460602
2020-04-30    1843007
2020-05-31    4171471
2020-06-30    5772802
2020-07-31    7684224
2020-08-31    7136506
2020-09-30    6684185
2020-10-31    7522627
2020-11-30    9225202
2020-12-31    8143547
2021-01-31    8527310
2021-02-28    6044941
2021-03-31    1366401
Freq: ME, Name: negativeIncrease, dtype: int64


### Step 3: Drop Rows Based on Specific Columns

In [185]:
df_clean = df.dropna(subset=['positive', 'negative','pending','hospitalizedCurrently','hospitalizedCumulative','inIcuCurrently','inIcuCumulative','onVentilatorCurrently'])
print(df_clean)

         date  states    positive    negative  pending  hospitalizedCurrently  \
0    20210307      56  28756489.0  74582825.0  11808.0                40199.0   
1    20210306      56  28714654.0  74450990.0  11783.0                41401.0   
2    20210305      56  28654639.0  74307155.0  12213.0                42541.0   
3    20210304      56  28585852.0  74035238.0  12405.0                44172.0   
4    20210303      56  28520365.0  73857281.0  11778.0                45462.0   
..        ...     ...         ...         ...      ...                    ...   
342  20200330      56    171965.0    416268.0  65369.0                15892.0   
343  20200329      56    150826.0    366103.0  65404.0                14055.0   
344  20200328      56    131143.0    324271.0  65698.0                12393.0   
345  20200327      56    111358.0    255808.0  59795.0                10887.0   
346  20200326      56     92143.0    210652.0  60131.0                 7805.0   

     hospitalizedCumulative

### Step 4: Drop non required Columns

In [194]:
df_clean.drop(columns=['recovered', 'total','posNeg'], inplace=True)
print(df_clean)

         date  states    positive    negative  pending  hospitalizedCurrently  \
0    20210307      56  28756489.0  74582825.0  11808.0                40199.0   
1    20210306      56  28714654.0  74450990.0  11783.0                41401.0   
2    20210305      56  28654639.0  74307155.0  12213.0                42541.0   
3    20210304      56  28585852.0  74035238.0  12405.0                44172.0   
4    20210303      56  28520365.0  73857281.0  11778.0                45462.0   
..        ...     ...         ...         ...      ...                    ...   
342  20200330      56    171965.0    416268.0  65369.0                15892.0   
343  20200329      56    150826.0    366103.0  65404.0                14055.0   
344  20200328      56    131143.0    324271.0  65698.0                12393.0   
345  20200327      56    111358.0    255808.0  59795.0                10887.0   
346  20200326      56     92143.0    210652.0  60131.0                 7805.0   

     hospitalizedCumulative

/var/folders/18/ygzmjvj90m72g8_1d1sfk1b80000gn/T/ipykernel_24787/1582743883.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean.drop(columns=['recovered', 'total','posNeg'], inplace=True)


### Step 5: sort data by new date column

In [202]:
df_clean = df_clean.sort_values(by='date_new', ascending=False)
df_clean.head(10)

,date,states,positive,negative,pending,hospitalizedCurrently,hospitalizedCumulative,inIcuCurrently,inIcuCumulative,onVentilatorCurrently,...,death,hospitalized,totalTestResults,lastModified,deathIncrease,hospitalizedIncrease,negativeIncrease,positiveIncrease,totalTestResultsIncrease,date_new
0,20210307,56,28756489.0,74582825.0,11808.0,40199.0,776361.0,8134.0,45475.0,2802.0,...,515151.0,776361.0,363825123,2021-03-07T24:00:00Z,842,726,131835,41835,1170059,2021-03-07
1,20210306,56,28714654.0,74450990.0,11783.0,41401.0,775635.0,8409.0,45453.0,2811.0,...,514309.0,775635.0,362655064,2021-03-06T24:00:00Z,1680,503,143835,60015,1430992,2021-03-06
2,20210305,56,28654639.0,74307155.0,12213.0,42541.0,775132.0,8634.0,45373.0,2889.0,...,512629.0,775132.0,361224072,2021-03-05T24:00:00Z,2221,2781,271917,68787,1744417,2021-03-05
3,20210304,56,28585852.0,74035238.0,12405.0,44172.0,772351.0,8970.0,45293.0,2973.0,...,510408.0,772351.0,359479655,2021-03-04T24:00:00Z,1743,1530,177957,65487,1590984,2021-03-04
4,20210303,56,28520365.0,73857281.0,11778.0,45462.0,770821.0,9359.0,45214.0,3094.0,...,508665.0,770821.0,357888671,2021-03-03T24:00:00Z,2449,2172,267001,66836,1406795,2021-03-03
5,20210302,56,28453529.0,73590280.0,11196.0,46388.0,768649.0,9465.0,45084.0,3169.0,...,506216.0,768649.0,356481876,2021-03-02T24:00:00Z,1728,1871,255779,54248,1343519,2021-03-02
6,20210301,56,28399281.0,73334501.0,11748.0,46738.0,766778.0,9595.0,44956.0,3171.0,...,504488.0,766778.0,355138357,2021-03-01T24:00:00Z,1241,1024,118077,48092,1154440,2021-03-01
7,20210228,56,28351189.0,73216424.0,11708.0,47352.0,765754.0,9802.0,44907.0,3245.0,...,503247.0,765754.0,353983917,2021-02-28T24:00:00Z,1051,879,203599,54349,1408422,2021-02-28
8,20210227,56,28296840.0,73012825.0,11731.0,48871.0,764875.0,10114.0,44875.0,3335.0,...,502196.0,764875.0,352575495,2021-02-27T24:00:00Z,1847,1428,205090,71245,1655179,2021-02-27
9,20210226,56,28225595.0,72807735.0,11945.0,51112.0,763447.0,10466.0,44791.0,3466.0,...,500349.0,763447.0,350920316,2021-02-26T24:00:00Z,2141,1868,276829,74857,1803309,2021-02-26
